In [0]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, concat
from pyspark.sql import functions as F
from datetime import datetime


In [0]:
schema_establishments = StructType([
    StructField("DocumentNumber", StringType(), True),
    StructField("CnpjOrder", StringType(), True),
    StructField("CnpjDv", StringType(), True),
    StructField("CompanyType", StringType(), True),
    StructField("TradingName", StringType(), True),
    StructField("TaxIdStatus",  StringType(), True),
    StructField("TaxIdUpdateDate",  DateType(), True),
    StructField("MotivoSituacaoCadastral",  StringType(), True),
    StructField("NomeCidadeExterior", StringType(), True),
    StructField("Pais", StringType(), True), 
    StructField("FoundedDate",  DateType(), True),
    StructField("CNAE", StringType(), True),
    StructField("CNAESecondary", StringType(), True),
    StructField("TipoLogradouro", StringType(), True),
    StructField("Address_Street", StringType(), True),
    StructField("Address_Number", StringType(), True),
    StructField("Address_Complement", StringType(), True),
    StructField("Address_District", StringType(), True),
    StructField("Address_PostalCode", StringType(), True), 
    StructField("Address_State", StringType(), True),
    StructField("Municipio", StringType(), True),
    StructField("DDD01", StringType(), True),
    StructField("Phone01", StringType(), True),
    StructField("DDD02", StringType(), True),
    StructField("Phone02", StringType(), True),
    StructField("FaxDDD", StringType(), True),
    StructField("Fax", StringType(), True),
    StructField("CompanyEmail", StringType(), True),
    StructField("SituacaoEspecial", StringType(), True),
    StructField("Data_Situacao_Especial", StringType(), True),
])

company_types_dict = {
    '1': 'Matriz',
    '2': 'Filial'
}
tax_id_types = {
    '01': 'Nula',
    '02': 'Ativa',
    '03': 'Suspensa',
    '04': 'Inapta',
    '08': 'Baixada'
}

SparkSession.builder.appName("establishments_silver").getOrCreate()

In [0]:
domains_df = spark.read.csv('/Volumes/workspace/cnpjs_schema/auxiliar/domains/KnowDomains.csv')

rows = domains_df.collect()
domains_list = []

for row in rows:
    domains_list.append(row['_c0'])

domains_list

In [0]:
domains_list

In [0]:
def get_month():
    today = datetime.now()
    actual_month = today.month 
    actual_year = today.year

    if today.month == 1:

        actual_month = 12
        actual_year = today.year - 1
        url_date = f'{actual_year}-{actual_month}'
    else:
        actual_month = today.month - 1
        url_date = f'{actual_year}-{actual_month:02d}'
        return(url_date)

url_date = get_month()

In [0]:
cnae_codes = pd.read_csv(f'/Volumes/workspace/cnpjs_schema/auxiliar/CNAE/{url_date}_Cnaes.csv', 
    encoding='latin1', 
    sep=';',
    dtype={ 'cnae_code': str, 'cnae':str },
    names=['cnae_code', 'cnae'],
    index_col = ['cnae_code']
    )
cnae_replace = cnae_codes.to_dict()
cnae_replace = cnae_replace['cnae']

In [0]:
domains_df = domains_df.withColumn("_c0", F.upper(F.col("_c0")))

In [0]:
establishments_df = spark.read \
    .format("csv") \
    .schema(schema_establishments) \
    .option("header", "true") \
    .option("sep", ";") \
    .option("DateFormat", "yyyyMMdd") \
    .load(f"/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/bronze")


In [0]:
establishments_df = establishments_df.replace(tax_id_types, subset=["TaxIdStatus"])
establishments_df = establishments_df.filter(F.col("TaxIdStatus") == 'Ativa')
establishments_df = establishments_df.withColumn("email_extraido", F.upper(F.get(F.split("CompanyEmail", "@"), 1)))

In [0]:
df_joined = establishments_df.join(
    domains_df, 
    F.col("email_extraido") == domains_df['_c0'], "left"
)

establishments_df = df_joined.withColumn(
    "CompanyWebsite",
    F.when(
        F.col("_c0").isNull(),           
        F.col("CompanyEmail")           
    ).otherwise(F.lit(None))           
)

In [0]:

to_dropcolumns = ['CnpjOrder','CnpjDv', 'DDD01','Phone01','DDD02', 'Phone02','Ddd_Fax','FAX', 'FAXDDD', 'Pais','TipoLogradouro', 'Municipio', 'SituacaoEspecial', 'Data_Situacao_Especial', 'NomeCidadeExterior','MotivoSituacaoEspecial', 'MotivoSituacaoCadastral', 'email_extraido','_c0' ]

establishments_df =( establishments_df
.withColumn("CNPJ", F.concat(F.col("DocumentNumber"), F.col("CnpjOrder"), F.col("CnpjDv")))
.withColumn("CompanyPhoneNumber", F.concat(F.col("DDD01"), F.col("Phone01")))
.withColumn("CompanyPhoneNumber02", F.concat(F.col("DDD02"), F.col("Phone02")))
.replace(company_types_dict, subset=["CompanyType"])
.drop(*to_dropcolumns)
.withColumn("Address_Complement", F.regexp_replace(F.col("Address_Complement"), r"[^a-zA-Z0-9\s]", ""))
.withColumn("Address_Complement", F.regexp_replace(F.col("Address_Complement"), r"\s+", " "))
.withColumn("Address_Complement", F.trim(F.col("Address_Complement")))
.replace(cnae_replace, subset=["CNAE"])
.withColumn("CNPJ", F.trim(F.col("CNPJ")))
)






In [0]:
establishments_df.write.mode("overwrite").option("overwriteSchema", "True").format("delta").saveAsTable('workspace.establishments_silver.silver_establishments')